In [13]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm  # PyTorch Image Models (for ViT)

# Define device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load ViT model from timm (Pre-trained model)
class ViTModel(nn.Module):
    def __init__(self, num_classes):
        super(ViTModel, self).__init__()
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=True)
        self.vit.head = nn.Linear(self.vit.head.in_features, num_classes)  # Adjusting the final layer

    def forward(self, x):
        return self.vit(x)

# Define the transform for your input data
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # Normalizing to ImageNet pre-trained model
])

# Load your datasets
train_dataset = datasets.ImageFolder('./train', transform=transform)
test_dataset = datasets.ImageFolder('./test', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Initialize the model
num_classes = 3  # Replace this with the actual number of classes
model = ViTModel(num_classes=num_classes).to(device)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
def train(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    return running_loss / len(loader)

# Test loop
def test(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    accuracy = correct / total
    return running_loss / len(loader), accuracy

# Training the model
epochs = 20
for epoch in range(epochs):
    train_loss = train(model, train_loader, criterion, optimizer, device)
    test_loss, test_accuracy = test(model, test_loader, criterion, device)
    
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


Epoch 1/20, Train Loss: 1.1672, Test Loss: 0.5181, Test Accuracy: 0.8538
Epoch 2/20, Train Loss: 0.4541, Test Loss: 0.2148, Test Accuracy: 0.9142
Epoch 3/20, Train Loss: 0.2504, Test Loss: 0.1189, Test Accuracy: 0.9513
Epoch 4/20, Train Loss: 0.1726, Test Loss: 0.1066, Test Accuracy: 0.9629
Epoch 5/20, Train Loss: 0.1207, Test Loss: 0.0937, Test Accuracy: 0.9513
Epoch 6/20, Train Loss: 0.0974, Test Loss: 0.2655, Test Accuracy: 0.9397
Epoch 7/20, Train Loss: 0.1316, Test Loss: 0.0831, Test Accuracy: 0.9675
Epoch 8/20, Train Loss: 0.1183, Test Loss: 0.1671, Test Accuracy: 0.9490
Epoch 9/20, Train Loss: 0.0822, Test Loss: 0.0889, Test Accuracy: 0.9675
Epoch 10/20, Train Loss: 0.0392, Test Loss: 0.0592, Test Accuracy: 0.9791
Epoch 11/20, Train Loss: 0.0839, Test Loss: 0.0677, Test Accuracy: 0.9652
Epoch 12/20, Train Loss: 0.0674, Test Loss: 0.1050, Test Accuracy: 0.9652
Epoch 13/20, Train Loss: 0.0477, Test Loss: 0.1485, Test Accuracy: 0.9466
Epoch 14/20, Train Loss: 0.0236, Test Loss: 0.0